In [2]:
import pandas as pd
import numpy as np
import surprise
import os
os.chdir("C:/Users/PGCP-AI/ML/MachineLearning/Cases_Rec_Sys/Amazon Movie Ratings")

In [9]:
df = pd.read_csv("Amazon.csv")
df

,user_id,Movie1,Movie2,Movie3,Movie4,Movie5,Movie6,Movie7,Movie8,Movie9,...,Movie197,Movie198,Movie199,Movie200,Movie201,Movie202,Movie203,Movie204,Movie205,Movie206
0,A3R5OBKS7OM2IR,5.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AH3QC2PC1VTGP,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,A3LKP6WPMP9UKX,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AVIY68KEPQ5ZD,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,A1CV1WROP5KTTW,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4843,A1IMQ9WMFYKWH5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0
4844,A1KLIKPUF5E88I,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0
4845,A5HG6WFZLO10D,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0
4846,A3UU690TWXCG1X,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0


In [10]:
ratings = pd.melt(df, id_vars='user_id', 
                  var_name="item_id",value_name="rating")
ratings

,user_id,item_id,rating
0,A3R5OBKS7OM2IR,Movie1,5.0
1,AH3QC2PC1VTGP,Movie1,NaN
2,A3LKP6WPMP9UKX,Movie1,NaN
3,AVIY68KEPQ5ZD,Movie1,NaN
4,A1CV1WROP5KTTW,Movie1,NaN
...,...,...,...
998683,A1IMQ9WMFYKWH5,Movie206,5.0
998684,A1KLIKPUF5E88I,Movie206,5.0
998685,A5HG6WFZLO10D,Movie206,5.0
998686,A3UU690TWXCG1X,Movie206,5.0


In [11]:
ratings = ratings[ratings['rating'].notna()]

In [12]:
ratings

,user_id,item_id,rating
0,A3R5OBKS7OM2IR,Movie1,5.0
4848,A3R5OBKS7OM2IR,Movie2,5.0
9697,AH3QC2PC1VTGP,Movie3,2.0
14546,A3LKP6WPMP9UKX,Movie4,5.0
14547,AVIY68KEPQ5ZD,Movie4,5.0
...,...,...,...
998683,A1IMQ9WMFYKWH5,Movie206,5.0
998684,A1KLIKPUF5E88I,Movie206,5.0
998685,A5HG6WFZLO10D,Movie206,5.0
998686,A3UU690TWXCG1X,Movie206,5.0


In [13]:
lowest_rating = ratings['rating'].min()
highest_rating = ratings['rating'].max()
lowest_rating,highest_rating

(1.0, 5.0)

In [14]:
reader = surprise.Reader(rating_scale=(lowest_rating,highest_rating))
data = surprise.Dataset.load_from_df(df=ratings,reader=reader)

In [15]:
similarity_options = {'name':'cosine','user_based':True}
algo = surprise.KNNBasic(sim_options=similarity_options)
output = algo.fit(data.build_full_trainset())

Computing the cosine similarity matrix...
Done computing similarity matrix.


In [22]:
from surprise.model_selection import GridSearchCV
from surprise.model_selection.split import KFold
param_grid ={'k':[20,30,50,70,90],'user_based':[True,False]}
kfold= KFold(n_splits=5,shuffle=True,random_state=26)
# gs=GridSearchCV(surprise.KNNBasic,param_grid,measures=['rmse','mae'],cv=kfold)
gs=GridSearchCV(surprise.KNNWithZScore,param_grid,measures=['rmse','mae'],cv=kfold)
gs.fit(data)

Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computi

In [23]:
print(gs.best_score['rmse'])
print(gs.best_params['rmse'])

1.2018142692025235
{'k': 20, 'user_based': True}


In [21]:
user = 50
u_iid = ratings[ratings['user_id']==user]['item_id'].unique()
iids = ratings['item_id'].unique()
print('List of items rated by user:',u_iid)
print('No. of items rated by user {0} : {1} '.format(user,len(u_iid)))
iids_to_predict = np.setdiff1d(iids,u_iid)
print("Items not rated by the user or those items for which the expected ratings are to be predicted",iids_to_predict)

List of items rated by user: []
No. of items rated by user 50 : 0 
Items not rated by the user or those items for which the expected ratings are to be predicted ['Movie1' 'Movie10' 'Movie100' 'Movie101' 'Movie102' 'Movie103' 'Movie104'
 'Movie105' 'Movie106' 'Movie107' 'Movie108' 'Movie109' 'Movie11'
 'Movie110' 'Movie111' 'Movie112' 'Movie113' 'Movie114' 'Movie115'
 'Movie116' 'Movie117' 'Movie118' 'Movie119' 'Movie12' 'Movie120'
 'Movie121' 'Movie122' 'Movie123' 'Movie124' 'Movie125' 'Movie126'
 'Movie127' 'Movie128' 'Movie129' 'Movie13' 'Movie130' 'Movie131'
 'Movie132' 'Movie133' 'Movie134' 'Movie135' 'Movie136' 'Movie137'
 'Movie138' 'Movie139' 'Movie14' 'Movie140' 'Movie141' 'Movie142'
 'Movie143' 'Movie144' 'Movie145' 'Movie146' 'Movie147' 'Movie148'
 'Movie149' 'Movie15' 'Movie150' 'Movie151' 'Movie152' 'Movie153'
 'Movie154' 'Movie155' 'Movie156' 'Movie157' 'Movie158' 'Movie159'
 'Movie16' 'Movie160' 'Movie161' 'Movie162' 'Movie163' 'Movie164'
 'Movie165' 'Movie166' 'Movie167'

In [24]:
testset = [[user,iid,0.] for iid in iids_to_predict]
predictions = algo.test(testset)
exp_ratings = [ (predictions[i].iid, predictions[i].est) for i in range(0, len(predictions))]
df  = pd.DataFrame(exp_ratings, columns = ["iid", "est_rating"])
df.sort_values('est_rating',ascending=False)

,iid,est_rating
0,Movie1,4.3856
141,Movie40,4.3856
131,Movie31,4.3856
132,Movie32,4.3856
133,Movie33,4.3856
...,...,...
71,Movie163,4.3856
72,Movie164,4.3856
73,Movie165,4.3856
74,Movie166,4.3856
